## Disclaimer and Documentation of AI Usage
This project was completed with the assistance of AI models, including agentic AI coding models. The code included was written by these models. Models used include: Claude Sonnet 4.6 (through Claude Chat and Claude Code), Claude Opus 4.6 (through Claude Chat and Claude Code), Gemini 3.1 Pro (through Gemini CLI and the Gemini Web Interface), ChatGPT 5.2 Thinking (through the ChatGPT Web Interface), and GPT 5.2 Codex and GPT 5.3 Codex (through the Codex web interface and mobile application).
Due to the many iterative changes and updates made to the codebase by these various models, one cannot attribute a specific section of code to one AI model or prompt. For more granular detail on which models made specific changes, please see the history of commits and PRs by the various AI models located at this project's GitHub Repo: https://github.com/shahvezk79/sst_helper

## Authored by:
Shahvez Khan, Emily Hawkins, Michelle Niazi, and Khushi Gajeeban.


---
## Tool Access and Instructions

If you would like to access and test the SST search tool, please visit the following link: https://ssthelper-production.up.railway.app

To use the tool, first turn off development mode and ensure you are using cloud compute. Then, initialize the database, input your facts, and perform a search. Thank you for your time!

## Introduction

Self-represented litigants before Canada's Social Security Tribunal (SST) face a
difficult challenge: finding past decisions relevant to their situation among
thousands of published rulings. Traditional keyword search fails because legal
concepts can be expressed in many different ways — a claimant searching for
"fired for being late" needs to find decisions about "misconduct" and
"just cause."

This tool addresses that gap with a **three-stage pipeline** (two-stage retrieval + generation):

1. **Stage 1 — Semantic Search (Bi-Encoder):** Embeds both the user's query and
   all 17,000+ SST decisions into a shared vector space using
   [Qwen3-Embedding-8B](https://huggingface.co/Qwen/Qwen3-Embedding-8B), then
   retrieves the top 40 candidates by cosine similarity.

2. **Stage 2 — Cross-Encoder Reranking:** Scores each candidate with a
   cross-encoder model
   ([Qwen3-Reranker-8B](https://huggingface.co/Qwen/Qwen3-Reranker-8B)) that
   reads the query and document *together*, using a **section-aware packing**
   strategy to prioritize the most legally salient parts of each decision.

3. **Stage 3 — Case Card Generation:** Produces a plain-language four-section
   summary of the top result using
   [Qwen3-14B](https://huggingface.co/Qwen/Qwen3-14B), designed to be
   understandable by non-lawyers.

### Architecture

```
User Query
    │
    ▼
┌─────────────────────────────────┐
│  Stage 1: Bi-Encoder            │  Qwen3-Embedding-8B
│  Semantic Search                │  Cosine similarity → Top 40 candidates
└───────────────┬─────────────────┘
                │
                ▼
┌─────────────────────────────────┐
│  Stage 2: Cross-Encoder         │  Qwen3-Reranker-8B
│  Reranking                      │  Section-aware packing → Top 5 results
└───────────────┬─────────────────┘
                │
                ▼
┌─────────────────────────────────┐
│  Stage 3: Case Card Generation  │  Qwen3-14B
│  Plain-language summary         │  Issue / Key Facts / Test / Outcome
└─────────────────────────────────┘
```

### Data Source

Decisions are sourced from the
[A2AJ Canadian Case Law](https://huggingface.co/datasets/a2aj/canadian-case-law)
dataset on Hugging Face, which provides the full text of published SST rulings.
Pre-computed embeddings for the entire corpus are cached on Hugging Face to avoid
the expensive one-time encoding step.

---
## 2A. Pipeline Overview (Plain-Language 5-Box Visual)

```text
[1] User Query
     A claimant asks a question in everyday language.

            ↓

[2] Section-Aware Retrieval
     The system finds similar SST decisions and focuses on sections like
     Issue/Analysis/Conclusion rather than only metadata.

            ↓

[3] Reranking
     A stronger model re-scores candidates so the most legally relevant
     decisions move to the top.

            ↓

[4] Summarization with Citations
     The tool creates a short case card and points users to source decisions
     so claims are traceable.

            ↓

[5] Risk Labels + User View
     Results are presented with plain-language cautions (informational, not
     legal advice) to reduce over-reliance.
```

**Why this matters for non-technical readers:** each stage has one clear job, and trust/safety is integrated into the pipeline output rather than added at the end.


---
## 1. Setup

Install the required dependencies and configure the API key for cloud inference.

In [1]:
%pip install -q pandas pyarrow numpy huggingface_hub openai requests certifi

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging
import json
import os
import re
import ssl
from dataclasses import dataclass, field
from getpass import getpass
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    import certifi
except ImportError:
    certifi = None

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)-30s  %(levelname)-8s  %(message)s",
)
logger = logging.getLogger("sst_navigator")

# ── DeepInfra API key ────────────────────────────────────────────────
# Required for Stage 1 query embedding, Stage 2 reranking, and
# Stage 3 generation.  Free tier available at https://deepinfra.com
if not os.environ.get("DEEPINFRA_API_KEY"):
    os.environ["DEEPINFRA_API_KEY"] = getpass("Enter your DeepInfra API key: ")

print("Setup complete.")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


---
## 2. Configuration

All model identifiers, pipeline parameters, and API endpoints are defined here.
These constants are referenced throughout the pipeline.

In [3]:
# ── Model identifiers (DeepInfra cloud) ──────────────────────────────
DEEPINFRA_BASE_URL         = "https://api.deepinfra.com/v1/openai"
DEEPINFRA_EMBEDDING_MODEL  = "Qwen/Qwen3-Embedding-8B"
DEEPINFRA_RERANKER_MODEL   = "Qwen/Qwen3-Reranker-8B"
DEEPINFRA_RERANKER_ENDPOINT = (
    "https://api.deepinfra.com/v1/inference/Qwen/Qwen3-Reranker-8B"
)
DEEPINFRA_GENERATION_MODEL = "Qwen/Qwen3-14B"

# ── Pipeline parameters ─────────────────────────────────────────────
STAGE1_TOP_K   = 40     # Candidates from semantic search
STAGE2_TOP_K   = 5      # Final results after cross-encoder reranking
SNIPPET_LENGTH = 500    # Characters for the preview snippet

# Fast-mode reduces latency at a small quality cost
FAST_STAGE1_TOP_K = 20
FAST_STAGE2_TOP_K = 3

# ── Embedding parameters ────────────────────────────────────────────
EMBEDDING_INSTRUCTION = (
    "Represent this query for retrieving similar Canadian "
    "administrative legal tribunal decisions"
)
EMBEDDING_MAX_TOKENS       = 8192
FAST_EMBEDDING_MAX_TOKENS  = 4096

# Pre-computed embedding cache hosted on HuggingFace
EMBEDDING_CACHE_DIR        = ".cache/embeddings"
EMBEDDING_CACHE_REPO_ID    = "mystic63/sst-embeddings-cache"
EMBEDDING_CACHE_REPO_TYPE  = "dataset"
EMBEDDING_CACHE_FILE       = "sst_embeddings_qwen3_8b.npy"
EMBEDDING_METADATA_FILE    = "metadata.json"

# ── Reranker parameters ─────────────────────────────────────────────
RERANKER_INSTRUCTION = (
    "Given a legal query describing an appellant's situation, "
    "retrieve relevant Social Security Tribunal of Canada decisions"
)
RERANKER_MAX_TOKENS      = 8192
FAST_RERANKER_MAX_TOKENS = 2048

# ── Generation parameters ───────────────────────────────────────────
GENERATION_MAX_TOKENS      = 512
FAST_GENERATION_MAX_TOKENS = 256
GENERATION_MAX_CHARS       = 24_000
FAST_GENERATION_MAX_CHARS  = 12_000

# ── Data ─────────────────────────────────────────────────────────────
DEV_ROW_LIMIT = 500   # Row limit for fast development iteration

print("Configuration loaded.")

Configuration loaded.


---
## 3. Data Loading

SST decisions are loaded from the
[A2AJ Canadian Case Law](https://huggingface.co/datasets/a2aj/canadian-case-law)
dataset on Hugging Face. The dataset is distributed as a Parquet file and
contains the full text of every published SST ruling along with metadata
(case name, date, URL).

Key cleaning steps:
- Drop rows where the decision text is empty or null
- Verify that the expected columns exist
- Optionally trim to a subset for faster development

In [4]:
PARQUET_URL = (
    "https://huggingface.co/datasets/a2aj/canadian-case-law"
    "/resolve/main/SST/train.parquet"
)
REQUIRED_COLUMNS = ["name_en", "document_date_en", "url_en", "unofficial_text_en"]


def _is_ssl_cert_error(error: Exception) -> bool:
    """Check if an exception is caused by an SSL certificate error."""
    return "CERTIFICATE_VERIFY_FAILED" in str(error)


def _configure_certifi_ca_bundle() -> bool:
    """Use certifi's CA bundle as a fallback for HTTPS connections."""
    if certifi is None:
        return False
    ca_path = certifi.where()
    os.environ.setdefault("SSL_CERT_FILE", ca_path)
    os.environ.setdefault("REQUESTS_CA_BUNDLE", ca_path)
    ssl._create_default_https_context = lambda: ssl.create_default_context(
        cafile=ca_path
    )
    return True


def load_sst_decisions(max_rows: int | None = None) -> pd.DataFrame:
    """Load SST decisions from the A2AJ Hugging Face dataset.

    Args:
        max_rows: If set, only load the first N rows (useful for dev/testing).

    Returns:
        DataFrame with cleaned SST decisions.
    """
    logger.info(
        "Loading SST decisions from Hugging Face (%s rows)...",
        max_rows or "all",
    )
    try:
        df = pd.read_parquet(PARQUET_URL)
    except Exception as e:
        if _is_ssl_cert_error(e) and _configure_certifi_ca_bundle():
            logger.warning(
                "SSL certificate verification failed. "
                "Retrying with certifi CA bundle."
            )
            df = pd.read_parquet(PARQUET_URL)
        else:
            raise RuntimeError(
                "Could not load the SST dataset. "
                f"Check your internet connection.\n{e}"
            ) from e

    # Verify expected columns exist
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Dataset is missing expected columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    # Drop rows where the decision text is empty or null
    df = df.dropna(subset=["unofficial_text_en"])
    df = df[df["unofficial_text_en"].str.strip().astype(bool)]
    df = df.reset_index(drop=True)

    if max_rows is not None:
        df = df.head(max_rows)
        logger.info("Trimmed to %d rows for development.", len(df))

    logger.info("Loaded %d SST decisions.", len(df))
    return df

In [5]:
# Load the full dataset (set max_rows=DEV_ROW_LIMIT for faster iteration)
df = load_sst_decisions(max_rows=None)

print(f"Total decisions: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['document_date_en'].min()} to {df['document_date_en'].max()}")
print(f"Average text length: {df['unofficial_text_en'].str.len().mean():,.0f} characters")
df[["name_en", "document_date_en", "url_en"]].head(10)

2026-03-02 19:43:43,257  sst_navigator                   INFO      Loading SST decisions from Hugging Face (all rows)...
2026-03-02 19:44:09,957  sst_navigator                   INFO      Loaded 17334 SST decisions.


Total decisions: 17,334
Columns: ['dataset', 'citation_en', 'citation2_en', 'name_en', 'document_date_en', 'url_en', 'scraped_timestamp_en', 'unofficial_text_en', 'citation_fr', 'citation2_fr', 'name_fr', 'document_date_fr', 'url_fr', 'scraped_timestamp_fr', 'unofficial_text_fr', 'upstream_license']
Date range: 2013-03-08 00:00:00+00:00 to 2026-01-30 00:00:00+00:00
Average text length: 13,317 characters


,name_en,document_date_en,url_en
0,G. F. v. Canada Employment Insurance Commission,2015-05-24 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
1,M. D. v. Canada Employment Insurance Commission,2015-05-25 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
2,J. Z. v. Minister of Employment and Social Dev...,2015-09-11 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/cpp-rp...
3,J. T. v. Canada Employment Insurance Commission,2015-05-20 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
4,C. T. v. Canada Employment Insurance Commission,2015-05-20 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
5,C. N. v. Canada Employment Insurance Commission,2015-05-26 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
6,M. M. v. Canada Employment Insurance Commission,2015-05-22 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
7,H. B. v. Canada Employment Insurance Commission,2015-05-27 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...
8,I. R. v. Minister of Employment and Social Dev...,2015-09-04 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/cppd-r...
9,P. R. v. Canada Employment Insurance Commission,2015-05-29 00:00:00+00:00,https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/...


---
## 4. Section-Aware Document Parsing

SST decisions follow a standard structure with recognisable headings:
**Overview → Issue → The Law → Analysis → Conclusion**. Not every decision uses
the same headings, but the pattern is consistent enough to exploit.

Our section parser:
1. **Strips boilerplate** — removes metadata headers, table-of-contents lines,
   and trailing footnote blocks.
2. **Detects section headings** — uses a regex to find capitalised headings
   followed by paragraph markers like `[1]`, `[2]`, etc.
3. **Assigns a priority** to each section based on its heading, ranking the
   most legally salient content highest:

| Priority | Headings | Why |
|----------|----------|-----|
| 1 (highest) | Analysis, Law and Analysis | Contains the legal test and the tribunal's reasoning |
| 2 | Issue, What I have to decide | Frames the legal question |
| 3 | Conclusion, Decision | States the outcome |
| 4 | Overview, Introduction, Background | Context |
| 5 | The Law, Applicable Law | Statutory provisions (usually boilerplate) |
| 6 | Evidence, Submissions | Factual record |
| 7 | Everything else | Custom or unusual headings |

4. **Greedily packs** sections into a character budget for the reranker,
   starting with the highest-priority sections. Analysis sections receive
   special truncation: the head (which states the legal test) and the last
   two paragraphs (which contain the finding) are kept.

This approach ensures the cross-encoder sees the most decision-relevant text
even when the full document far exceeds the model's context window.

In [6]:
# ── Section data structure ───────────────────────────────────────────

class Section(NamedTuple):
    name: str       # heading text, e.g. "Analysis"
    text: str       # body text (heading + paragraphs)
    priority: int   # 1 = highest value for relevance scoring
    order: int      # original position in the document


# ── Priority mapping ─────────────────────────────────────────────────
# Ordered list of (compiled pattern, priority).  First match wins.

_PRIORITY_RULES: list[tuple[re.Pattern, int]] = [
    (re.compile(r"analysis|law and analysis", re.IGNORECASE), 1),
    (re.compile(
        r"issue|issues|what i have to decide|what i must decide"
        r"|what the .* must prove",
        re.IGNORECASE,
    ), 2),
    (re.compile(r"conclusion|decision", re.IGNORECASE), 3),
    (re.compile(r"overview|introduction|background", re.IGNORECASE), 4),
    (re.compile(
        r"the law|applicable law|what the law says|legal framework",
        re.IGNORECASE,
    ), 5),
    (re.compile(
        r"evidence|submissions|what the .* says",
        re.IGNORECASE,
    ), 6),
]

_DEFAULT_PRIORITY = 7


def _heading_priority(name: str) -> int:
    """Return the priority tier (1 = highest) for a section heading."""
    for pattern, priority in _PRIORITY_RULES:
        if pattern.search(name):
            return priority
    return _DEFAULT_PRIORITY


# ── Boilerplate stripping ────────────────────────────────────────────

def _strip_boilerplate(text: str) -> str:
    """Remove metadata header and trailing footnote block."""
    # Strip metadata before "Decision Content"
    dc_idx = text.find("Decision Content")
    if dc_idx != -1:
        text = text[dc_idx + len("Decision Content"):]

    # Skip "On this page" TOC line
    toc_idx = text.find("On this page")
    if toc_idx != -1:
        after_toc = text.find("\n", toc_idx)
        if after_toc != -1:
            next_newline = text.find("\n", after_toc + 1)
            if next_newline != -1:
                text = text[next_newline + 1:]
            else:
                text = text[after_toc + 1:]
        else:
            text = text[toc_idx + len("On this page"):]

    # Skip "Reasons and decision" bridge line (older decisions)
    rd_match = re.match(r"\s*Reasons and decision\s*", text)
    if rd_match:
        text = text[rd_match.end():]

    # Strip trailing footnote block
    fn_match = re.search(r"\nFootnote 1\n", text)
    if fn_match:
        text = text[:fn_match.start()]

    return text.strip()


# ── Section splitting ────────────────────────────────────────────────
# Matches a heading followed (possibly with whitespace) by a [N]
# paragraph marker.  The heading starts with a capital letter and
# contains letters, spaces, hyphens, dashes, commas, apostrophes,
# and slashes.

_HEADING_RE = re.compile(
    r"^([A-Z][A-Za-z][A-Za-z \-\u2013\u2014,/\']{0,80}?)"
    r"\s*"
    r"(?=\[\d+\])",
    re.MULTILINE,
)


def parse_sections(text: str) -> list[Section]:
    """Parse an SST decision into prioritised sections.

    Returns a list of Section objects sorted by priority (ascending)
    with document order as tiebreaker within the same tier.
    """
    cleaned = _strip_boilerplate(text)
    if not cleaned:
        return []

    matches = list(_HEADING_RE.finditer(cleaned))
    if not matches:
        # No headings found — return the entire text as one section.
        return [Section(
            name="body", text=cleaned,
            priority=_DEFAULT_PRIORITY, order=0,
        )]

    sections: list[Section] = []
    for i, m in enumerate(matches):
        name = m.group(1).strip()
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned)
        body = cleaned[start:end].strip()
        sections.append(Section(
            name=name,
            text=body,
            priority=_heading_priority(name),
            order=i,
        ))

    return sections


# ── Greedy packing ───────────────────────────────────────────────────

_PARAGRAPH_RE = re.compile(r"\[\d+\]")
_SECTION_SEP = "\n\n"


def _truncate_analysis(text: str, budget: int) -> str:
    """Truncate an Analysis section keeping the head and last 2 paragraphs.

    The head typically contains the legal test the tribunal applies,
    while the final paragraphs contain the finding.  Keeping both gives
    the cross-encoder the most decision-relevant signal.
    """
    if len(text) <= budget:
        return text

    tail_budget = budget // 5
    head_budget = budget - tail_budget - len(" [...] ")

    para_matches = list(_PARAGRAPH_RE.finditer(text))
    if len(para_matches) >= 3:
        tail_start = para_matches[-2].start()
        tail = text[tail_start:]
        if len(tail) > tail_budget:
            tail = tail[:tail_budget]
    else:
        tail = ""
        head_budget = budget

    head = text[:head_budget]
    if tail:
        return head + " [...] " + tail
    return head


def pack_for_reranker(text: str, char_budget: int) -> str:
    """Pack an SST decision into a character budget for the reranker.

    Extracts sections, prioritises legally salient content (Analysis,
    Issue, Conclusion), and greedily fills the budget.  Falls back to
    head-truncation when section parsing fails.
    """
    sections = parse_sections(text)
    if not sections:
        return text[:char_budget]

    # If the entire cleaned text fits, just return it in document order.
    total = (
        sum(len(f"[{s.name}] ") + len(s.text) for s in sections)
        + len(_SECTION_SEP) * (len(sections) - 1)
    )
    if total <= char_budget:
        return _SECTION_SEP.join(
            f"[{s.name}] {s.text}"
            for s in sorted(sections, key=lambda s: s.order)
        )

    # Sort by (priority, original order) — highest-value first.
    ordered = sorted(sections, key=lambda s: (s.priority, s.order))

    packed: list[tuple[int, str]] = []  # (original order, formatted text)
    remaining = char_budget

    for s in ordered:
        if remaining <= 0:
            break

        label = f"[{s.name}] "
        label_len = len(label)
        body_budget = remaining - label_len - len(_SECTION_SEP)
        if body_budget <= 0:
            break

        if len(s.text) <= body_budget:
            packed.append((s.order, label + s.text))
            remaining -= len(label) + len(s.text) + len(_SECTION_SEP)
        else:
            if s.priority == 1:
                truncated = _truncate_analysis(s.text, body_budget)
            else:
                truncated = s.text[:body_budget]
            packed.append((s.order, label + truncated))
            remaining -= len(label) + len(truncated) + len(_SECTION_SEP)

    # Reassemble in original document order.
    packed.sort(key=lambda t: t[0])
    return _SECTION_SEP.join(text for _, text in packed)


print("Section parser defined.")

Section parser defined.


In [7]:
# Demo: parse a real SST decision into sections
sample_decision = df.iloc[0]["unofficial_text_en"]
sections = parse_sections(sample_decision)

print(f"Decision: {df.iloc[0]['name_en']}")
print(f"Sections found: {len(sections)}\n")
print(f"{'Section':<40} {'Priority':>8}  {'Length':>8}")
print("-" * 60)
for s in sorted(sections, key=lambda s: (s.priority, s.order)):
    print(f"{s.name:<40} {s.priority:>8}  {len(s.text):>8,}")

# Show how packing works with a small budget
packed = pack_for_reranker(sample_decision, char_budget=4000)
print(f"\n--- Packed for reranker (budget=4000 chars, used={len(packed):,}) ---")
print(packed[:1500] + "\n[...]" if len(packed) > 1500 else packed)

Decision: G. F. v. Canada Employment Insurance Commission
Sections found: 7

Section                                  Priority    Length
------------------------------------------------------------
Analysis                                        1     4,542
Issue                                           2       191
Conclusion                                      3        40
Introduction                                    4       769
The law                                         5     1,483
Evidence                                        6     3,641
Submissions                                     6     1,203

--- Packed for reranker (budget=4000 chars, used=3,998) ---
[Issue] Issue [5] Whether an indefinite disqualification should be imposed on the Appellant according to sections 29 and 30 of the Act, because he lost his employment by reason of his own misconduct.

[Analysis] Analysis [23] Subsection 30(1) of the Act provides for an indefinite disqualification when a claimant loses h

---
## 5. Stage 1 — Semantic Search (Bi-Encoder)

The first stage uses a **bi-encoder** architecture: the query and every document
are independently embedded into the same vector space, and candidates are
retrieved by cosine similarity.

### How it works

1. **Pre-computed document embeddings** are loaded from a cache on Hugging Face.
   Encoding 17,000+ decisions is expensive (~hours on a GPU), so we do it once
   and cache the results as a numpy `.npy` file alongside a `metadata.json`
   that records the URL ordering.

2. **At query time**, only the short user query is embedded via the DeepInfra
   API (Qwen3-Embedding-8B), wrapped with a task-specific instruction:
   > *"Represent this query for retrieving similar Canadian administrative
   > legal tribunal decisions"*

3. **Cosine similarity** between the query vector and all document vectors
   is computed as a single matrix multiplication (since all vectors are
   unit-normalised, dot product = cosine similarity). The top 40 candidates
   are returned.

### Embedding quality safeguards

- **Sanitisation:** Any NaN/Inf values in embedding vectors (rare but possible
  with quantised models) are replaced with zeros.
- **L2 normalisation:** Performed in float64 to avoid overflow when squaring
  large float32 values, then downcast back to float32 for efficient dot product.
- **Alignment:** The cached vectors may be in a different order than the
  DataFrame rows (the cache-building script sorts by text length for efficient
  batching). The `align_to_urls` method reorders vectors to match the DataFrame.

In [8]:
# ── Embedding utility functions ──────────────────────────────────────

def _sanitize_embedding_batch(result: np.ndarray) -> None:
    """Zero out rows containing NaN/Inf in *result* in-place.

    Rows replaced with zeros will produce a cosine similarity of 0
    and rank last in search results.
    """
    bad_rows = ~np.isfinite(result).all(axis=1)
    if bad_rows.any():
        n_bad = int(bad_rows.sum())
        logger.warning(
            "%d of %d embedding vectors contain NaN/Inf — "
            "replacing with zeros.",
            n_bad, result.shape[0],
        )
        result[bad_rows] = 0.0


def _l2_normalize_rows(arr: np.ndarray) -> np.ndarray:
    """Return a copy of *arr* with every row L2-normalised.

    Zero-norm rows (e.g. previously sanitised bad vectors) are left
    as zeros rather than producing NaN from 0/0.

    The norm is computed in float64 to avoid overflow when squaring
    large float32 values (e.g. 1e20² exceeds float32 max).
    """
    arr64 = arr.astype(np.float64)
    norms = np.linalg.norm(arr64, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return (arr64 / norms).astype(np.float32)


def _sanitize_and_normalize_rows(arr: np.ndarray) -> np.ndarray:
    """Return a sanitized + unit-normalized copy of *arr*."""
    clean = np.nan_to_num(arr, copy=True, nan=0.0, posinf=0.0, neginf=0.0)
    return _l2_normalize_rows(clean)


# ── SemanticSearcher class ───────────────────────────────────────────

class SemanticSearcher:
    """Embeds queries and searches a pre-computed document embedding index.

    Uses the DeepInfra API (Qwen3-Embedding-8B) for query encoding and
    pre-computed cached vectors for the document corpus.
    """

    def __init__(self):
        self._doc_embeddings: np.ndarray | None = None
        self._cache_urls: list[str] | None = None

    # -- Persistent cache ------------------------------------------------

    def load_embeddings_cache(
        self,
        cache_dir: str = EMBEDDING_CACHE_DIR,
        repo_id: str = EMBEDDING_CACHE_REPO_ID,
        repo_type: str = EMBEDDING_CACHE_REPO_TYPE,
        embeddings_file: str = EMBEDDING_CACHE_FILE,
        metadata_file: str = EMBEDDING_METADATA_FILE,
    ) -> bool:
        """Download and load cached embeddings from HuggingFace.

        The embedding vectors (.npy) and metadata (URL ordering, .json)
        are downloaded if not already present locally.
        """
        base = Path(cache_dir)
        base.mkdir(parents=True, exist_ok=True)
        emb_path = base / embeddings_file
        meta_path = base / metadata_file

        # Download from HuggingFace if missing locally
        if not emb_path.exists() or not meta_path.exists():
            try:
                hf_hub_download(
                    repo_id=repo_id,
                    filename=embeddings_file,
                    repo_type=repo_type,
                    local_dir=str(base),
                    local_dir_use_symlinks=False,
                )
                hf_hub_download(
                    repo_id=repo_id,
                    filename=metadata_file,
                    repo_type=repo_type,
                    local_dir=str(base),
                    local_dir_use_symlinks=False,
                )
                logger.info("Downloaded embedding cache from %s", repo_id)
            except Exception as exc:
                logger.warning(
                    "Could not download embedding cache: %s", exc
                )

        if not emb_path.exists() or not meta_path.exists():
            return False

        # Parse metadata to get the URL ordering
        try:
            raw = json.loads(meta_path.read_text(encoding="utf-8"))
        except Exception as exc:
            logger.error("Invalid metadata JSON: %s", exc)
            return False

        if isinstance(raw, list):
            self._cache_urls = [str(u) for u in raw]
        elif isinstance(raw, dict) and "url_en" in raw:
            self._cache_urls = [str(u) for u in raw["url_en"]]
        else:
            logger.error("Unexpected metadata format.")
            return False

        self._doc_embeddings = np.load(
            emb_path, allow_pickle=False,
        ).astype(np.float32, copy=False)

        # Validate consistency
        if self._doc_embeddings.shape[0] != len(self._cache_urls):
            logger.error(
                "Mismatch: %d vectors vs %d URLs.",
                self._doc_embeddings.shape[0], len(self._cache_urls),
            )
            self._doc_embeddings = None
            self._cache_urls = None
            return False

        # Deduplicate URLs (keep first occurrence)
        n_unique = len(set(self._cache_urls))
        if n_unique < len(self._cache_urls):
            seen: set[str] = set()
            keep: list[int] = []
            for i, url in enumerate(self._cache_urls):
                if url not in seen:
                    seen.add(url)
                    keep.append(i)
            self._doc_embeddings = self._doc_embeddings[np.array(keep)]
            self._cache_urls = [self._cache_urls[i] for i in keep]

        logger.info(
            "Loaded embedding cache — %d vectors, dim=%d",
            self._doc_embeddings.shape[0], self._doc_embeddings.shape[1],
        )
        return True

    def align_to_urls(self, target_urls: list[str]) -> list[int]:
        """Reorder cached embeddings to match *target_urls*.

        The cached vectors may be in a different order than the
        DataFrame rows.  This method reorders them so that
        embedding row i corresponds to DataFrame row i.

        Returns the indices of target_urls that had a cache match.
        """
        if self._doc_embeddings is None or self._cache_urls is None:
            raise RuntimeError("Call load_embeddings_cache() first.")

        cache_lookup = {url: i for i, url in enumerate(self._cache_urls)}

        matched_target: list[int] = []
        matched_cache: list[int] = []
        for target_idx, url in enumerate(target_urls):
            cache_idx = cache_lookup.get(url)
            if cache_idx is not None:
                matched_target.append(target_idx)
                matched_cache.append(cache_idx)

        if not matched_cache:
            raise RuntimeError("No cached embeddings match the target URLs.")

        self._doc_embeddings = self._doc_embeddings[np.array(matched_cache)]
        self._cache_urls = None  # no longer needed

        # Sanitise + re-normalise cached vectors
        self._doc_embeddings = _sanitize_and_normalize_rows(
            self._doc_embeddings
        )

        logger.info(
            "Aligned embeddings: %d of %d target URLs matched.",
            len(matched_target), len(target_urls),
        )
        return matched_target

    # -- Query embedding (DeepInfra API) ---------------------------------

    def _embed_query(self, text: str) -> np.ndarray:
        """Embed a single query string via the DeepInfra API."""
        from openai import OpenAI

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        client = OpenAI(api_key=api_key, base_url=DEEPINFRA_BASE_URL)
        response = client.embeddings.create(
            input=[text],
            model=DEEPINFRA_EMBEDDING_MODEL,
            encoding_format="float",
        )
        vector = np.array(
            [response.data[0].embedding], dtype=np.float32,
        )
        _sanitize_embedding_batch(vector)
        return vector

    # -- Search ----------------------------------------------------------

    def search(
        self,
        query: str,
        top_k: int = STAGE1_TOP_K,
    ) -> list[tuple[int, float]]:
        """Return the top-K (index, score) pairs for a query.

        The query is wrapped with the retrieval instruction before
        encoding.  Similarity is computed via dot product (vectors
        are already unit-normalised).
        """
        if self._doc_embeddings is None:
            raise RuntimeError("Load embeddings first.")
        if self._doc_embeddings.shape[0] == 0:
            return []

        # Wrap query with the task instruction
        formatted_query = (
            f"Instruct: {EMBEDDING_INSTRUCTION}\n"
            f"Query:{query}"
        )
        q_vec = self._embed_query(formatted_query)
        q_vec = _sanitize_and_normalize_rows(q_vec)

        # Cosine similarity via dot product
        with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
            scores = self._doc_embeddings @ q_vec.T
        scores = scores.squeeze(-1)
        np.nan_to_num(
            scores, copy=False, nan=-1.0, posinf=-1.0, neginf=-1.0,
        )

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in top_indices]

    @property
    def has_embeddings(self) -> bool:
        return (
            self._doc_embeddings is not None
            and self._doc_embeddings.shape[0] > 0
        )


print("SemanticSearcher defined.")

SemanticSearcher defined.


In [9]:
# Initialize the searcher and load pre-computed embeddings
searcher = SemanticSearcher()

print("Downloading embedding cache from HuggingFace...")
if not searcher.load_embeddings_cache():
    raise RuntimeError(
        "Failed to load embedding cache from HuggingFace. "
        "Check your internet connection and try re-running this cell."
    )

# Align cached vectors with our DataFrame rows
target_urls = df["url_en"].astype(str).tolist()
matched_indices = searcher.align_to_urls(target_urls)

# Trim DataFrame to only rows with cached embeddings
if len(matched_indices) < len(df):
    df = df.iloc[matched_indices].reset_index(drop=True)
    print(f"Trimmed DataFrame to {len(df):,} rows with cached embeddings.")

print(f"\nEmbedding index ready:")
print(f"  Documents: {searcher._doc_embeddings.shape[0]:,}")
print(f"  Dimensions: {searcher._doc_embeddings.shape[1]}")

2026-03-02 20:54:39,276  sst_navigator                   INFO      Loaded embedding cache — 17334 vectors, dim=4096


2026-03-02 20:54:40,103  sst_navigator                   INFO      Aligned embeddings: 17334 of 17334 target URLs matched.



Embedding index ready:
  Documents: 17,334
  Dimensions: 4096


In [10]:
# Run a sample semantic search
sample_query = (
    "I was fired for being late to work multiple times "
    "and my EI benefits were denied for misconduct"
)

print(f"Query: {sample_query!r}\n")
stage1_hits = searcher.search(sample_query, top_k=STAGE1_TOP_K)

print(f"Stage 1 returned {len(stage1_hits)} candidates.\n")
print(f"{'Rank':<6} {'Score':>8}  {'Decision Name'}")
print("-" * 70)
for rank, (idx, score) in enumerate(stage1_hits[:10], 1):
    row = df.iloc[idx]
    name = row["name_en"][:50]
    print(f"{rank:<6} {score:>8.4f}  {name}")
print("...")

Query: 'I was fired for being late to work multiple times and my EI benefits were denied for misconduct'



2026-03-02 20:54:52,867  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/embeddings "HTTP/1.1 200 OK"


Stage 1 returned 40 candidates.

Rank      Score  Decision Name
----------------------------------------------------------------------
1        0.7835  BB v Canada Employment Insurance Commission
2        0.7794  CM v Canada Employment Insurance Commission
3        0.7753  AK v Canada Employment Insurance Commission
4        0.7730  TP v Canada Employment Insurance Commission
5        0.7656  EG v Canada Employment Insurance Commission
6        0.7649  DM v Canada Employment Insurance Commission
7        0.7602  X v Canada Employment Insurance Commission and GC
8        0.7549  BA v Canada Employment Insurance Commission
9        0.7548  CM v Canada Employment Insurance Commission
10       0.7542  X v Canada Employment Insurance Commission and CS
...


---
## 6. Stage 2 — Cross-Encoder Reranking

The bi-encoder from Stage 1 is fast but approximate: it encodes the query
and documents *independently*, so it cannot model fine-grained interactions
between them.

Stage 2 uses a **cross-encoder** (Qwen3-Reranker-8B) that reads the query
and each candidate document *together* and produces a single relevance score.
This is much more accurate but too slow to run over the full corpus — hence the
two-stage approach.

### Section-aware packing

SST decisions can be very long (10,000+ words), but the reranker has a limited
context window. Rather than naively truncating from the beginning, we use the
**section parser** from Section 4 to intelligently pack the most legally
salient sections into the character budget:

1. Parse the decision into sections
2. Sort by priority (Analysis > Issue > Conclusion > ...)
3. Greedily pack sections until the budget is full
4. Reassemble in original document order

This ensures the cross-encoder sees the tribunal's reasoning and conclusion
rather than procedural metadata.

In [11]:
class Reranker:
    """Cross-encoder reranker using the DeepInfra API.

    Scores query-document pairs using Qwen3-Reranker-8B and returns
    the top-K candidates sorted by relevance.
    """

    def _score_batch(
        self, query: str, documents: list[str],
    ) -> list[float]:
        """Score all documents in one API call via DeepInfra."""
        import requests as req

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        # Prepend the task instruction to the query
        formatted_query = f"{RERANKER_INSTRUCTION}\n{query}"

        resp = req.post(
            DEEPINFRA_RERANKER_ENDPOINT,
            headers={
                "Authorization": f"bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "queries": [formatted_query],
                "documents": documents,
            },
            timeout=120,
        )
        resp.raise_for_status()
        data = resp.json()

        scores = data["scores"]
        # API may return [[s1, s2, ...]] (one list per query) or [s1, s2, ...]
        if scores and isinstance(scores[0], list):
            scores = scores[0]
        return [float(s) for s in scores]

    def rerank(
        self,
        query: str,
        candidates: list[dict],
        top_k: int = STAGE2_TOP_K,
        max_tokens: int = RERANKER_MAX_TOKENS,
    ) -> list[dict]:
        """Score and re-sort candidates by cross-encoder relevance.

        Each candidate dict must contain at least 'text' and 'index' keys.
        Returns the top_k candidates sorted by descending reranker score.
        """
        logger.info("Reranking %d candidates...", len(candidates))

        # Pack each document's most salient sections within budget
        documents = []
        for cand in candidates:
            doc_text = pack_for_reranker(
                cand["text"], char_budget=max_tokens * 4,
            )
            documents.append(doc_text)

        # Score all candidates in a single API call
        scores = self._score_batch(query, documents)

        for cand, score in zip(candidates, scores):
            cand["reranker_score"] = score

        ranked = sorted(
            candidates,
            key=lambda c: c["reranker_score"],
            reverse=True,
        )
        return ranked[:top_k]


print("Reranker defined.")

Reranker defined.


In [12]:
# Build candidate dicts from Stage 1 hits
candidates = []
for idx, score in stage1_hits:
    row = df.iloc[idx]
    candidates.append({
        "index": idx,
        "name": row.get("name_en", "Unnamed"),
        "date": str(row.get("document_date_en", "")),
        "url": row.get("url_en", ""),
        "text": row["unofficial_text_en"],
        "stage1_score": score,
    })

# Run the cross-encoder reranker
reranker = Reranker()
top_results = reranker.rerank(
    sample_query, candidates,
    top_k=STAGE2_TOP_K,
    max_tokens=RERANKER_MAX_TOKENS,
)

print(f"Stage 2 returned {len(top_results)} results.\n")
print(f"{'Rank':<6} {'Reranker':>10} {'Stage1':>10}  {'Decision Name'}")
print("-" * 75)
for rank, r in enumerate(top_results, 1):
    print(
        f"{rank:<6} {r['reranker_score']:>10.4f} "
        f"{r['stage1_score']:>10.4f}  "
        f"{r['name'][:45]}"
    )

2026-03-02 20:55:28,442  sst_navigator                   INFO      Reranking 40 candidates...


Stage 2 returned 5 results.

Rank     Reranker     Stage1  Decision Name
---------------------------------------------------------------------------
1          0.9999     0.7794  CM v Canada Employment Insurance Commission
2          0.9998     0.7362  J. J. v. Canada Employment Insurance Commissi
3          0.9997     0.7305  DM v Canada Employment Insurance Commission
4          0.9997     0.7463  E. B. v. Canada Employment Insurance Commissi
5          0.9996     0.7835  BB v Canada Employment Insurance Commission


---
## 7. Stage 3 — Case Card Generation

The final stage generates a **plain-language summary** of the top-ranked
decision, designed to help self-represented litigants quickly understand
the key aspects of a case without reading the full text.

The summary follows a fixed four-section format:

- **Issue:** What legal question was the tribunal deciding?
- **Key Facts:** What were the most important facts?
- **Test Applied:** What legal test or criteria did the tribunal use?
- **Outcome:** What did the tribunal decide, and why?

The generation model (Qwen3-14B on DeepInfra) is prompted with a system
message that enforces this structure and instructs the model to use simple
language and not invent facts.

Output is post-processed to remove any `<think>...</think>` reasoning
blocks that the model may produce (Qwen3 models use these for internal
chain-of-thought).

In [13]:
# ── Case card system prompt ──────────────────────────────────────────

_CASE_CARD_SYSTEM = (
    "You are a legal-aid assistant. Read the tribunal decision below "
    "and produce a plain-language summary a self-represented person "
    "can understand. Output EXACTLY four sections with these "
    "headings:\n\n"
    "**Issue:** (What legal question was the tribunal deciding?)\n"
    "**Key Facts:** (What were the most important facts?)\n"
    "**Test Applied:** (What legal test or criteria did the tribunal "
    "use?)\n"
    "**Outcome:** (What did the tribunal decide, and why?)\n\n"
    "Be concise. Use simple language. Do not invent facts."
)

_THINK_BLOCK_RE = re.compile(
    r"<think>.*?</think>", re.IGNORECASE | re.DOTALL,
)


def _sanitize_case_card_output(text: str) -> str:
    """Remove hidden reasoning blocks, keep the user-facing summary."""
    sanitized = _THINK_BLOCK_RE.sub("", text)

    expected = (
        "**Issue:**", "**Key Facts:**",
        "**Test Applied:**", "**Outcome:**",
    )
    positions = [
        sanitized.find(s) for s in expected if s in sanitized
    ]
    if positions:
        sanitized = sanitized[min(positions):]

    return sanitized.strip()


class CaseCardGenerator:
    """Generate a structured plain-language summary of an SST decision.

    Uses the DeepInfra API (Qwen3-14B) for generation.
    """

    def generate_case_card(
        self,
        decision_text: str,
        max_tokens: int = GENERATION_MAX_TOKENS,
        max_chars: int = GENERATION_MAX_CHARS,
    ) -> str:
        """Return a formatted case-card string for the given decision."""
        from openai import OpenAI

        api_key = os.environ.get("DEEPINFRA_API_KEY")
        if not api_key:
            raise RuntimeError("Set DEEPINFRA_API_KEY in your environment.")

        client = OpenAI(
            api_key=api_key,
            base_url=DEEPINFRA_BASE_URL,
        )
        truncated = decision_text[:max_chars]

        resp = client.chat.completions.create(
            model=DEEPINFRA_GENERATION_MODEL,
            messages=[
                {"role": "system", "content": _CASE_CARD_SYSTEM},
                {
                    "role": "user",
                    "content": (
                        "Here is the tribunal decision:\n\n"
                        + truncated
                    ),
                },
            ],
            max_tokens=max_tokens,
            temperature=0.3,
        )
        return _sanitize_case_card_output(
            resp.choices[0].message.content,
        )


print("CaseCardGenerator defined.")

CaseCardGenerator defined.


In [14]:
# Generate a case card for the top-ranked result
generator = CaseCardGenerator()

top_decision_text = top_results[0]["text"]
print(f"Generating case card for: {top_results[0]['name']}\n")

case_card = generator.generate_case_card(
    top_decision_text,
    max_tokens=GENERATION_MAX_TOKENS,
    max_chars=GENERATION_MAX_CHARS,
)

print(case_card)

Generating case card for: CM v Canada Employment Insurance Commission



2026-03-02 20:56:13,224  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"


**Issue:**  
Was the person (Appellant) fired because of their own misconduct, which would make them ineligible for Employment Insurance (EI) benefits?  

**Key Facts:**  
The Appellant was fired for being late to work on November 14, 2023, to celebrate their wife’s birthday. They had a history of being late and received warnings from their employer. The employer said they had a strict policy requiring employees to report late arrivals at least 30 minutes before their shift. The Appellant claimed they couldn’t afford gas and only received one formal warning, but the employer said they had multiple warnings and breached the policy by calling in too late.  

**Test Applied:**  
The tribunal used the legal definition of “misconduct” under the EI Act. Misconduct means the person’s actions were intentional or reckless, knowing they could lead to termination. The tribunal checked if the Appellant knew


---
## 8. Full Pipeline

The pipeline orchestrator ties all three stages together into a single
`search()` call. It manages the data lifecycle (loading, indexing) and
dispatches to each stage sequentially.

The pipeline supports a **fast mode** that reduces the number of candidates
at each stage and shortens token budgets, trading a small amount of quality
for significantly faster response times.

| Parameter | Normal | Fast |
|-----------|--------|------|
| Stage 1 candidates | 40 | 20 |
| Stage 2 results | 5 | 3 |
| Reranker max tokens | 8,192 | 2,048 |
| Generation max tokens | 512 | 256 |
| Generation max chars | 24,000 | 12,000 |

In [15]:
# ── Pipeline data structures ─────────────────────────────────────────

@dataclass
class SearchResult:
    """A single ranked decision returned to the user."""
    rank: int
    name: str
    date: str
    url: str
    reranker_score: float
    snippet: str
    full_text: str


@dataclass
class PipelineOutput:
    """Everything returned after a search."""
    case_card: str | None = None
    results: list[SearchResult] = field(default_factory=list)


# ── Pipeline orchestrator ────────────────────────────────────────────

class SSTNavigatorPipeline:
    """End-to-end three-stage RAG pipeline for SST decisions."""

    def __init__(self, dev_mode: bool = False, fast_mode: bool = False):
        self.dev_mode = dev_mode
        self.fast_mode = fast_mode
        self._df: pd.DataFrame | None = None
        self._searcher = SemanticSearcher()
        self._reranker = Reranker()
        self._generator = CaseCardGenerator()

    def _active_params(self) -> dict:
        """Return runtime parameters based on quality/speed mode."""
        if self.fast_mode:
            return {
                "stage1_top_k": FAST_STAGE1_TOP_K,
                "stage2_top_k": FAST_STAGE2_TOP_K,
                "reranker_max_tokens": FAST_RERANKER_MAX_TOKENS,
                "generation_max_tokens": FAST_GENERATION_MAX_TOKENS,
                "generation_max_chars": FAST_GENERATION_MAX_CHARS,
            }
        return {
            "stage1_top_k": STAGE1_TOP_K,
            "stage2_top_k": STAGE2_TOP_K,
            "reranker_max_tokens": RERANKER_MAX_TOKENS,
            "generation_max_tokens": GENERATION_MAX_TOKENS,
            "generation_max_chars": GENERATION_MAX_CHARS,
        }

    def load_data(self) -> int:
        """Load the SST dataset and return the row count."""
        max_rows = DEV_ROW_LIMIT if self.dev_mode else None
        self._df = load_sst_decisions(max_rows=max_rows)
        return len(self._df)

    def build_index(self) -> None:
        """Download cached embeddings and align with the DataFrame."""
        if self._df is None:
            raise RuntimeError("Call load_data() first.")

        if not self._searcher.load_embeddings_cache():
            raise RuntimeError("Could not load embedding cache.")

        target_urls = self._df["url_en"].astype(str).tolist()
        matched = self._searcher.align_to_urls(target_urls)

        if len(matched) < len(target_urls):
            self._df = self._df.iloc[matched].reset_index(drop=True)
            logger.warning(
                "Trimmed to %d rows with cached embeddings.",
                len(matched),
            )

    @property
    def is_ready(self) -> bool:
        return (
            self._df is not None and self._searcher.has_embeddings
        )

    def search(
        self,
        query: str,
        include_case_card: bool = True,
    ) -> PipelineOutput:
        """Run the full three-stage search pipeline.

        1. Semantic search (bi-encoder)  → top candidates
        2. Rerank (cross-encoder)        → top results
        3. Generate case card for #1     → plain-language summary
        """
        if not self.is_ready:
            raise RuntimeError(
                "Pipeline not initialised. "
                "Call load_data() + build_index()."
            )

        params = self._active_params()

        # ── Stage 1: Semantic search ─────────────────────────────
        hits = self._searcher.search(
            query, top_k=params["stage1_top_k"],
        )
        logger.info("Stage 1 returned %d candidates.", len(hits))

        if not hits:
            return PipelineOutput(
                case_card="No similar cases found.",
                results=[],
            )

        # Build candidate dicts for the reranker
        candidates = []
        for idx, score in hits:
            row = self._df.iloc[idx]
            candidates.append({
                "index": idx,
                "name": row.get("name_en", "Unnamed"),
                "date": str(row.get("document_date_en", "")),
                "url": row.get("url_en", ""),
                "text": row["unofficial_text_en"],
                "stage1_score": score,
            })

        # ── Stage 2: Reranking ───────────────────────────────────
        top_results = self._reranker.rerank(
            query, candidates,
            top_k=params["stage2_top_k"],
            max_tokens=params["reranker_max_tokens"],
        )
        logger.info("Stage 2 returned %d results.", len(top_results))

        if not top_results:
            return PipelineOutput(
                case_card="No relevant cases after reranking.",
                results=[],
            )

        # ── Stage 3: Optional case-card generation ───────────────
        case_card: str | None = None
        if include_case_card:
            case_card = self._generator.generate_case_card(
                top_results[0]["text"],
                max_tokens=params["generation_max_tokens"],
                max_chars=params["generation_max_chars"],
            )

        # ── Assemble output ──────────────────────────────────────
        results = []
        for rank, r in enumerate(top_results, 1):
            results.append(SearchResult(
                rank=rank,
                name=r["name"],
                date=r["date"],
                url=r["url"],
                reranker_score=r["reranker_score"],
                snippet=r["text"][:SNIPPET_LENGTH],
                full_text=r["text"],
            ))

        return PipelineOutput(case_card=case_card, results=results)


print("SSTNavigatorPipeline defined.")

SSTNavigatorPipeline defined.


In [ ]:
# Instead of re-loading data and embeddings (already done above),
# we wire the existing searcher + DataFrame into a pipeline instance.

pipeline = SSTNavigatorPipeline(dev_mode=False, fast_mode=False)
pipeline._df = df
pipeline._searcher = searcher  # reuse the already-loaded embedder

print(f"Pipeline ready: {pipeline.is_ready}")
print(f"Decisions indexed: {len(df):,}\n")

# ── Run a full search ────────────────────────────────────────────────
query = (
    "I have chronic back pain and depression that prevent me from working. My CPP disability application was denied."
)
print(f"Query: {query!r}\n")

output = pipeline.search(query, include_case_card=True)

# ── Display the case card ────────────────────────────────────────────
if output.case_card:
    print("=" * 70)
    print("CASE CARD (Top Result Summary)")
    print("=" * 70)
    print(output.case_card)
    print()

# ── Display ranked results ───────────────────────────────────────────
print("=" * 70)
print("RANKED RESULTS")
print("=" * 70)
for r in output.results:
    print(f"\n#{r.rank}  {r.name}")
    print(f"    Date:  {r.date}")
    print(f"    Score: {r.reranker_score:.4f}")
    print(f"    URL:   {r.url}")
    print(f"    Snippet: {r.snippet[:200]}...")

Pipeline ready: True
Decisions indexed: 17,334

Query: 'I have chronic back pain and depression that prevent me from working. My CPP disability application was denied.'



2026-03-02 20:56:58,167  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 20:56:58,438  sst_navigator                   INFO      Stage 1 returned 40 candidates.
2026-03-02 20:56:58,441  sst_navigator                   INFO      Reranking 40 candidates...
2026-03-02 20:57:11,915  sst_navigator                   INFO      Stage 2 returned 5 results.
2026-03-02 20:57:20,530  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"


CASE CARD (Top Result Summary)
**Issue:**  
Was the person correctly denied a disability pension under the Canada Pension Plan because the tribunal did not fully consider all of their medical conditions and how they affected their ability to work?  

**Key Facts:**  
The person had multiple health issues, including chronic pain, fibromyalgia, depression, back pain, shoulder problems, a knee injury, and medication side effects. The original tribunal decision only considered some of these conditions (like back and shoulder pain) but ignored others (like knee pain and depression). The person argued this incomplete analysis made the decision unfair. The government agreed the original decision may have missed key evidence.  

**Test Applied:**  
The tribunal checked whether the original decision properly considered all the evidence, including how the person’s combined physical and mental health issues prevented them from working regularly. It focused on whether the original tribunal’s analy

### Additional Query Example

Let's run a second query to demonstrate the pipeline's versatility across
different benefit types and legal questions.

In [18]:
query2 = (
    "I quit my job because my employer was harassing me and creating an unsafe work environment. EI said it was my own misconduct."
)
print(f"Query: {query2!r}\n")

output2 = pipeline.search(query2, include_case_card=True)

if output2.case_card:
    print("=" * 70)
    print("CASE CARD")
    print("=" * 70)
    print(output2.case_card)
    print()

print("=" * 70)
print("RANKED RESULTS")
print("=" * 70)
for r in output2.results:
    print(f"\n#{r.rank}  {r.name}")
    print(f"    Date:  {r.date}")
    print(f"    Score: {r.reranker_score:.4f}")
    print(f"    URL:   {r.url}")

Query: 'I quit my job because my employer was harassing me and creating an unsafe work environment. EI said it was my own misconduct.'



2026-03-02 20:59:24,155  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/embeddings "HTTP/1.1 200 OK"
2026-03-02 20:59:24,446  sst_navigator                   INFO      Stage 1 returned 40 candidates.
2026-03-02 20:59:24,451  sst_navigator                   INFO      Reranking 40 candidates...
2026-03-02 20:59:38,273  sst_navigator                   INFO      Stage 2 returned 5 results.
2026-03-02 20:59:48,552  httpx                           INFO      HTTP Request: POST https://api.deepinfra.com/v1/openai/chat/completions "HTTP/1.1 200 OK"


CASE CARD
**Issue:**  
Was the claimant disqualified from Employment Insurance (EI) benefits because he lost his job due to his own misconduct, or was his job loss caused by unsafe working conditions?  

**Key Facts:**  
The claimant worked for an employer, X, but had a heated argument at work. He claimed he was harassed and refused to return to the office until the employer investigated. The employer later dismissed him for refusing to follow instructions. The claimant applied

RANKED RESULTS

#1  JA v Canada Employment Insurance Commission
    Date:  2020-04-06 00:00:00+00:00
    Score: 0.9998
    URL:   https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/en/item/510095/index.do

#2  JT v Canada Employment Insurance Commission
    Date:  2023-11-28 00:00:00+00:00
    Score: 0.9996
    URL:   https://decisions.sst-tss.gc.ca/sst-tss/ei-ae/en/item/525268/index.do

#3  J. A. v. Canada Employment Insurance Commission
    Date:  2018-12-27 00:00:00+00:00
    Score: 0.9995
    URL:   https://deci

---
## 9. Conclusion

This notebook demonstrated a complete **three-stage RAG pipeline** for searching
Social Security Tribunal of Canada decisions using semantic AI:

1. **Data Loading** — 17,000+ SST decisions loaded from the A2AJ dataset on
   Hugging Face.

2. **Section-Aware Parsing** — Decisions are parsed into prioritised sections
   (Analysis, Issue, Conclusion, etc.) so that the most legally relevant
   content is always surfaced to the reranker, even for very long documents.

3. **Bi-Encoder Semantic Search** — Pre-computed embeddings enable sub-second
   retrieval of the top 40 candidates by cosine similarity, capturing semantic
   meaning rather than just keyword overlap.

4. **Cross-Encoder Reranking** — A more powerful model scores each candidate
   with full query-document attention, narrowing the field to the 5 most
   relevant decisions.

5. **Case Card Generation** — A plain-language summary makes the top result
   immediately understandable to self-represented litigants.

### Potential Future Work

- **Benefit-type filtering:** Pre-classify decisions by benefit area (EI, CPP,
  OAS) to allow scoped searches.
- **Citation graph:** Link decisions that cite each other to surface lines of
  authority.
- **Explainability:** Highlight which passages in a decision drove the reranker
  score.
- **Multilingual support:** Extend to French-language SST decisions.

---

> **Disclaimer:** This is an educational research tool and does not constitute
> legal advice. Decisions retrieved may not reflect the current state of the
> law. Please consult a licensed professional or your local legal-aid clinic
> for legal guidance.

---
## 10. Ethical and Professional Considerations

*Disclaimer: This piece of writing was edited via Grammarly.*

**Ethical and professional risks specific to this tool (and mitigations):**

Several risks are specific to a "cases like mine" tool designed for self-represented users because the system ranks past decisions by semantic relevance and then presents condensed case cards.

**Overreliance and "false confidence" in results.** Self-represented litigants may treat highly ranked results as predictive of their own outcome, especially when those results are presented cleanly, quickly, and in plain language. The Canadian Judicial Council has emphasized that self-represented parties face distinct barriers in navigating legal processes without counsel, which increases the risk that a tool like this will be treated as a substitute for legal judgment rather than as a research aid.

*Mitigations:* (a) strong interface language that the tool does not provide legal advice; (b) prominent links to the full SST decision and a short verification checklist; (c) clear display of decision dates and benefit type (EI/CPP/OAS) so users do not assume cross-program transferability; and (d) presenting a broader legal landscape rather than only seemingly favourable cases, such as by showing both a similar successful case and a similar unsuccessful one.

**Hallucinated, incomplete, or oversimplified case cards.** Even when generation is grounded in retrieved documents, large language models can still produce fluent but inaccurate, incomplete, or overconfident summaries. The case-card stage is therefore the highest-risk component in our pipeline because it converts a full decision into a short takeaway that users may treat as authoritative.

*Mitigations:* (a) keep outputs structured and short; (b) include extractive snippets from the decision, not only paraphrase; (c) require paragraph anchors where feasible so users can immediately verify key claims against the original decision; and (d) frame the case card as triage, meaning "is this worth reading?", rather than as a substitute for reading the decision itself. This concern remains even in retrieval-augmented systems: retrieval can improve factual grounding and provenance relative to parametric-only generation, but it does not eliminate error.

**Currency of law and "authority blindness."** Semantic relevance does not tell a user whether a decision is old, affected by legislative change, or displaced by later authority. That matters here because SST guidance expressly tells users to check whether a decision was reviewed by a court and whether it was overturned, and it explains that federal court decisions reviewing SST decisions are binding on SST members.

*Mitigations:* (a) display decision dates prominently and allow filtering or weighting by recency; (b) provide a "court-reviewed" or "precedent" flag where available; and (c) encourage noting up as a standard legal research step, with longer-term integration of noting-up signals if feasible. This is not just a generic AI risk. It is a predictable consequence of ranking by semantic relevance alone.

**Misinterpretation of retrieval scores.** In semantic retrieval systems, high-ranked matches can be driven by shared legal tests, repeated tribunal phrasing, or broad issue overlap even when the facts that actually determine the outcome are materially different. Strong-looking matches can therefore create misplaced confidence.

*Mitigations:* treat ranking scores as relative relevance signals rather than reliability guarantees, avoid presenting them as a "percent match," and provide short "why this matched" explanations, such as shared issue terms plus fact-pattern cues, so users can assess relevance more critically.

**Privacy risks from bulk processing and user queries.** Even where source decisions are public, digitization and repackaging can intensify privacy risks by making sensitive information easier to search, aggregate, and redistribute. The SST states that it removes names and other identifying information before publication and keeps personal information only where it is important to explaining the reasons for the decision. A tool that surfaces, ranks, and summarizes these decisions therefore creates an added risk if users enter their own identifying details into queries.

*Mitigations:* (a) warn users not to enter names, SINs, addresses, or other identifiers; (b) avoid storing or logging queries; (c) minimize excerpted text in case cards; and (d) rely on links to official decisions for full context rather than reproducing extensive passages.

---
## 11. Access to Justice

*Disclaimer: This piece of writing was edited via Grammarly.*

**Why this problem matters (and why "better search" is an access-to-justice intervention):**

Access to justice barriers are not limited to court fees or the availability of counsel. They also include practical barriers to locating, understanding, and using legal information. The Supreme Court of Canada has described access to justice as a systemic challenge tied to the rule of law. The Action Committee on Access to Justice in Civil and Family Matters likewise stresses that meaningful improvement requires concrete, system-level reform and usable public-facing tools, not merely putting materials online. The Canadian Judicial Council's Statement of Principles on Self-Represented Litigants and Accused Persons similarly recognizes that justice institutions must respond to the distinct information needs of people navigating legal processes without counsel.

Our project treats legal research friction as an access-to-justice problem in a tribunal context. SST decisions are public, but keyword search can fail when a user's ordinary-language description of a problem does not track the legal or tribunal language used in past decisions. The SST itself encourages users to search SST and court decisions to understand how the law works, understand how SST members make decisions, and find cases like theirs that may support arguments in an appeal. Our tool aims to make that step more usable for non-lawyers by combining semantic retrieval with short case cards that orient users to potentially relevant decisions, rather than functioning as a substitute for legal advice.

---

**Word count (excluding code and Appendix A): 2782 words**

---
## Appendix A — Validation Testing

*Note: This appendix is excluded from the main word count.*

*Disclaimer: The validation testing questions generated with Claude Sonnet 4.6 and the prompts by ChatGPT based on the GPT-5.3 model (OpenAI). Answers for the testing criteria were refined using ChatGPT based on the GPT-5.3 model (OpenAI) and edited via Grammarly.*

---

### Testing Part 1: Initial Testing

---

### Scenario 1: CPP Disability — Back Pain and Depression

> I had to stop working because of really bad back pain and depression. The tribunal said I might still be able to do some kind of lighter job so they denied my CPP disability.

#### Stage 1: Semantic Search (Bi-encoder Retrieval)

**Relevance of Candidate Set**
- *Do the top retrieved decisions relate to the user's described situation?*
  - Yes. All retrieved decisions involve CPP disability claims where the Tribunal assessed whether the claimant could still work despite medical conditions. This matches the user's scenario about being denied benefits because the tribunal believed they could do lighter work.
- *Does the tool correctly handle colloquial/plain-language queries (Ex: "my boss fired me unfairly" → EI appeals)?*
  - Yes, the user described their situation using non-legal language ("really bad back pain," "lighter job"), but the system successfully matched this to the legal concept of residual work capacity under the CPP severe disability test.

**Recall Quality**
- *Are semantically similar but legally distinct decisions appropriately distinguished (e.g., CPP retirement vs. CPP disability)?*
  - The results are all CPP disability qualification decisions, which is the correct program area. No unrelated EI or OAS cases appeared, suggesting the system distinguished between similar but legally different tribunal matters.

#### Stage 2: Cross-encoder Reranking

**Ranking Accuracy**
- *Does rank 1 represent the most fact-similar decision to the query?*
  - The top-ranked case appears the most factually similar to the scenario, focusing on whether serious health problems still allowed some work capacity. However, all of the cases were very similar factually.
- *Are decisions with similar facts but opposite outcomes both surfaced (so the user sees the full legal landscape)?*
  - Yes, and they were not clustered together, signifying that the system ranked the decisions based on factual similarity rather than outcome type.

**Score Calibration**
- *Do higher-scoring decisions feel meaningfully more relevant and/or different than lower-scoring ones?*
  - The scores were very close: 99.8%, 99.8%, 99.5%, 99.4%, 99.0%. This suggests all cases were highly relevant, but the small differences make it harder to see why one case ranked above another.

**Length of Time**
- *How long does it take to retrieve and rank decisions?*
  - 23 Seconds

#### Stage 3: Case Card Generation

**Accuracy**
- *Does the Issue section correctly identify the legal question decided in the source decision?*
  - The case card accurately identifies the legal issue as whether the applicant had a severe and prolonged disability by the end of the MQP (December 31, 2009) in order to qualify for CPP disability benefits.
- *Do the Key Facts reflect facts actually present in the decision text, with no hallucinated details?*
  - The key facts reflect the main details from the decision, including the applicant's back injury, depression, car accidents, limited treatment, and some continued functional ability (such as sitting, driving, and performing light tasks).
- *Does the Test Applied section correctly describe the legal standard used?*
  - The test applied correctly describes the CPP disability standard requiring a disability that is both severe and prolonged, and it appropriately notes that the tribunal considered the applicant's personal circumstances, medical evidence, and efforts to seek treatment or employment.
- *Does the Outcome section accurately reflect whether the appeal was allowed or dismissed?*
  - The outcome is also accurately reported: the tribunal dismissed the appeal because the evidence did not show the applicant was incapable of regularly pursuing substantially gainful employment, so he failed to meet the CPP's "severe" disability criteria.

**Plain Language Accessibility**
- *Can a self-represented litigant with no legal background understand the case card without needing to look up terminology?*
  - The case card is written in clear and accessible language that a self-represented litigant could understand and the explanation of the CPP disability test is simple without being overly technical.
- *Is the summary concise enough to be skimmed quickly (a few sentences per section)?*
  - The structure (Issue, Key Facts, Test Applied, Outcome) makes the decision easy to scan.

**Faithfulness to Source**
- *Does the case card avoid adding legal conclusions not present in the original decision?*
  - The summary appears faithful to the original decision. It does not introduce new legal reasoning or conclusions beyond what the tribunal stated.
- *Are quotes or paraphrases traceable back to the decision text?*
  - The facts and reasoning presented in the card are traceable to the decision and are presented in a neutral way.

**Length of Time**
- *How long does it take to generate the summary card?*
  - 14 seconds

#### Fast Mode Comparison Criteria

**Length of Time**
- *How long does it take to retrieve and rank decisions?* — 6 seconds
- *How long does it take to generate the summary card?* — 28 seconds

**Result Relevance**
- *Does the top case returned in fast mode address the same legal issue as the top case in regular mode?*
  - Yes. Both top results address the CPP disability severity test, specifically whether a claimant's medical conditions prevented them from regularly pursuing substantially gainful employment.
- *Does the fast mode still return factually similar decisions to the user's scenario?*
  - Yes. The fast mode cases involve claimants with serious health problems where the tribunal assessed whether they still had some capacity to work, which closely matches the user's scenario.
- *Are the three cases returned in fast mode included in the results of the regular mode?*
  - Two of the three cases were also returned in regular mode. One case was not in the regular-mode top five but still addresses the same legal issue.
- *Are the three cases returned in fast mode all reasonably relevant, or does the reduced candidate set introduce weaker results?*
  - All three cases appear reasonably relevant to the scenario and changing the mode did not introduce clearly irrelevant decisions.

**Coverage of Legal Landscape**
- *Does the reduced list of three cases omit any clearly important or highly relevant decision that appeared in regular mode?*
  - No clearly essential decision appears to be omitted. However, reducing the list from five to three cases slightly limits the range of examples available to the user.
- *If the regular mode showed cases with different outcomes, does fast mode still surface that variation?*
  - There was no variation in outcome, but the fast mode results still include cases showing different tribunal reasoning about work capacity, which helps illustrate how the severity test is applied in different circumstances.

**Summary Quality**
- *Is the summary generated in fast mode as accurate as the summary in regular mode?*
  - Yes. The fast-mode summary appears equally accurate in describing the tribunal decision.
- *Does the fast-mode summary still correctly describe: the legal issue, the key facts, the test applied, and the outcome?*
  - Yes. The summary correctly identifies the CPP disability eligibility issue, the relevant facts about the claimant's injuries and medical conditions, the severe and prolonged disability test, and the dismissal of the appeal.
- *Is the fast-mode summary clear and understandable for a non-lawyer, similar to the regular mode case card?*
  - Yes. The summary is written in clear and accessible language, and the structured format makes it easy for a non-lawyer to understand the key points of the decision.

#### Comparison against a legal database search: CanLII

**Initial search:** "CPP disability back pain depression tribunal lighter work"

The top results were all decisions from the Social Security Tribunal of Canada relating to CPP disability claims. For example, cases such as P.G. v. Minister of Employment and Social Development and L.G. v. Minister of Employment and Social Development involved claimants experiencing back pain and depression, with the Tribunal assessing whether they retained the capacity to work despite their conditions. The excerpts from these cases referenced the absence of severe medical evidence preventing work, as well as the possibility of performing lighter or part time work. Overall, the results were highly relevant to the scenario, even though the search was written in plain, non-legal language, and they consistently reflected the key issue of whether the claimant could pursue substantially gainful employment.

**Refined search:** "CPP disability severe prolonged work capacity tribunal decision."

The results from this search were more mixed and included decisions from different benefit programs, such as Employment Insurance and Old Age Security, in addition to CPP. For example, cases like MG v. Canada Employment Insurance Commission focused on eligibility requirements for EI benefits, while AM v. Minister of Employment and Social Development addressed Old Age Security eligibility and residency requirements. Although these decisions involved benefit entitlement and tribunal analysis, they were not directly focused on CPP disability or the severe and prolonged disability test. This made the results less targeted than those from the initial search.

**Reflection:**
This comparison shows that using plain language in the initial search actually produced more precise results than the refined search using formal legal terminology. While the refined search incorporated correct legal concepts, it also introduced broader terms that apply across multiple benefit programs, which reduced the specificity of the results. This suggests that effective searching on CanLII depends not only on using legal language, but on using sufficiently precise terms tied to a specific statutory context. In contrast, the AI system is able to interpret plain language and map it directly onto the correct legal framework without requiring the user to refine their query, making it more efficient and accessible for users without legal expertise.

---

### Scenario 2: Old Age Security — Residency Requirement

> I applied for Old Age Security but they told me I didn't live in Canada long enough to qualify for the full pension. I lived here for many years but also spent time living in another country.

#### Stage 1: Semantic Search (Bi-encoder Retrieval)

**Relevance of Candidate Set**
- *Do the top retrieved decisions relate to the user's described situation?*
  - Yes. All retrieved decisions relate to Old Age Security (OAS) residency requirements, which is the core issue raised in the prompt. The cases involve claimants who lived in Canada for some periods but also spent time abroad, requiring the Tribunal to determine whether they met the minimum residency requirement for OAS benefits.
- *Does the tool correctly handle colloquial/plain-language queries (Ex: "my boss fired me unfairly" → EI appeals)?*
  - Yes, the prompt described the issue informally ("didn't live in Canada long enough," "spent time living in another country"), and the system successfully matched it to the legal concept of OAS residency eligibility.

**Recall Quality**
- *Are semantically similar but legally distinct decisions appropriately distinguished (e.g., CPP retirement vs. CPP disability)?*
  - The results are all OAS cases dealing with residency, which indicates the system correctly distinguished this issue from other SST programs such as CPP disability or Employment Insurance. No unrelated benefit types appeared in the results, suggesting the retrieval model successfully separated semantically similar but legally distinct tribunal matters.

#### Stage 2: Cross-encoder Reranking

**Ranking Accuracy**
- *Does rank 1 represent the most fact-similar decision to the query?*
  - The top-ranked decision (M.H. v MESD) appears consistent with the user's scenario, as it involves the Tribunal determining whether the claimant met the OAS residency requirement despite periods spent outside Canada.
  - The remaining decisions address similar fact patterns, indicating that the reranking stage prioritized cases involving mixed residence in Canada and abroad.
- *Are decisions with similar facts but opposite outcomes both surfaced (so the user sees the full legal landscape)?*
  - There was only one decision where the appeal was not dismissed, but it was only allowed in part.

**Score Calibration**
- *Do higher-scoring decisions feel meaningfully more relevant and/or different than lower-scoring ones?*
  - The similarity scores were extremely close, suggesting that the model considered all five decisions highly relevant, though the small differences between scores make the ranking distinctions less meaningful.

**Length of Time**
- *How long does it take to retrieve and rank decisions?*
  - 13 seconds

#### Stage 3: Case Card Generation

**Accuracy**
- *Does the Issue section correctly identify the legal question decided in the source decision?*
  - The case card correctly identifies the legal issue as whether the applicant had a reasonable chance of success in obtaining leave to appeal the decision denying a full OAS pension and GIS.
- *Do the Key Facts reflect facts actually present in the decision text, with no hallucinated details?*
  - The key facts accurately summarize the tribunal's reasoning about the applicant's time spent outside Canada and the 40-year residency requirement beginning at age 18.
- *Does the Test Applied section correctly describe the legal standard used?*
  - The test applied is also correctly described. The summary explains that leave to appeal can only be granted if there is a legal error, factual error, or breach of procedural fairness in the original decision.
- *Does the Outcome section accurately reflect whether the appeal was allowed or dismissed?*
  - The outcome is accurate: the tribunal refused leave to appeal because the applicant did not demonstrate any reviewable error.

**Plain Language Accessibility**
- *Can a self-represented litigant with no legal background understand the case card without needing to look up terminology?*
  - The case card is clear and understandable for a non-lawyer, and the explanation of the appeal test is written in simple terms.
- *Is the summary concise enough to be skimmed quickly (a few sentences per section)?*
  - The structure (Issue, Key Facts, Test Applied, Outcome) allows the decision to be skimmed quickly.

**Faithfulness to Source**
- *Does the case card avoid adding legal conclusions not present in the original decision?*
  - The summary appears faithful to the decision and does not introduce legal conclusions beyond what the tribunal stated. The reasoning and outcome described in the case card are consistent with the decision's discussion of the OAS residency requirement and the leave-to-appeal standard.

**Length of Time**
- *How long does it take to generate the summary card?*
  - 15 seconds

#### Fast Mode Comparison Criteria

**Length of Time**
- *How long does it take to retrieve and rank decisions?* — 5 seconds
- *How long does it take to generate the summary card?* — 11 seconds

**Result Relevance**
- *Does the top case returned in fast mode address the same legal issue as the top case in regular mode?*
  - Yes. Both the fast-mode and regular-mode top cases address Old Age Security residency requirements, specifically whether the claimant met the residency threshold needed to qualify for OAS benefits.
- *Does the fast mode still return factually similar decisions to the user's scenario?*
  - Yes. The fast-mode cases involve claimants who lived in Canada for some periods but also spent time abroad, requiring the tribunal to determine whether they satisfied the OAS residency rules.
- *Are the three cases returned in fast mode included in the results of the regular mode?*
  - Yes. All three fast-mode cases were also included in the regular-mode results.
- *Are the three cases returned in fast mode all reasonably relevant, or does the reduced candidate set introduce weaker results?*
  - All three cases remain strongly relevant to the user's scenario. Reducing the candidate set to three decisions did not introduce clearly weaker or unrelated results.

**Coverage of Legal Landscape**
- *Does the reduced list of three cases omit any clearly important or highly relevant decision that appeared in regular mode?*
  - No clearly critical decisions appear to be omitted.
- *If the regular mode showed cases with different outcomes, does fast mode still surface that variation?*
  - The same variation occurred where one appeal was not dismissed, but it was only allowed in part.

**Summary Quality**
- *Is the summary generated in fast mode as accurate as the summary in regular mode?*
  - Yes. The fast-mode summary appears accurate and consistent with the tribunal decision.
- *Does the fast-mode summary still correctly describe: the legal issue, the key facts, the test applied, and the outcome?*
  - Yes. The summary correctly identifies the OAS residency issue, the claimant's history of living in Canada and abroad, the residency test under the OAS Act, and the outcome allowing eligibility for OAS and GIS from an earlier date.
- *Is the fast-mode summary clear and understandable for a non-lawyer, similar to the regular mode case card?*
  - Yes. The summary is written in clear, plain language, and the structured format makes the decision easy for a non-lawyer to understand.

#### Comparison against a legal database search: CanLII

**Initial Search:** "OAS didn't live in Canada long enough pension denied."

The top results were all decisions from the Social Security Tribunal of Canada relating to Old Age Security eligibility and residency requirements. For example, cases such as TC v. Minister of Employment and Social Development involved claimants being denied an OAS pension because they had not lived in Canada long enough, while other cases addressed whether time spent living abroad could count toward residency or qualify someone for a partial pension. Overall, the results were highly relevant to the scenario and consistently reflected the key issue of whether the claimant met the residency requirement for OAS benefits.

**Refined Search:** "Old Age Security residency requirement tribunal Canada years lived abroad"

The results of this search included decisions from both the Federal Court and the Social Security Tribunal of Canada relating to Old Age Security and residency requirements. For example, Stachowski v. Canada (Attorney General) addressed how residency is defined under the Old Age Security Act, while cases such as BA v. Minister of Employment and Social Development and Minister of Employment and Social Development v. LK focused on whether claimants met the minimum residency requirements to qualify for an OAS pension, particularly in situations involving time spent abroad. The excerpts from these decisions emphasized concepts such as "ordinary residence," time spent outside Canada, and the requirement of a minimum number of years of residence to receive full or partial benefits. Overall, the results remained closely connected to the issue of OAS eligibility and residency, though they included a broader mix of case types and levels of court.

**Reflection:**
These results show that the search was effective in identifying the correct legal issue but also introduced a wider range of decisions beyond the Social Security Tribunal, including higher court interpretations of residency under the Old Age Security Act. While this adds useful legal context, it may make it more difficult for a user to quickly identify factually similar cases. Compared to the AI system, which consistently returns highly tailored tribunal decisions, searching on CanLII can produce relevant but more varied results, requiring the user to further filter and interpret the cases to find those most closely aligned with their specific situation.

---

### Scenario 3: EI — Voluntary Leaving

> I quit my job because my workplace became really stressful and I felt like I had no choice. Service Canada said I'm not eligible for EI because I left voluntarily.

#### Stage 1: Semantic Search (Bi-encoder Retrieval)

**Relevance of Candidate Set**
- *Do the top retrieved decisions relate to the user's described situation?*
  - Yes. All retrieved decisions involve Employment Insurance claims where the claimant left their job voluntarily and the tribunal had to determine whether they had "just cause" for leaving. This closely matches the user's scenario, which describes quitting a job due to a stressful work environment and then being denied EI benefits.
- *Does the tool correctly handle colloquial/plain-language queries (Ex: "my boss fired me unfairly" → EI appeals)?*
  - Yes. The prompt uses informal wording ("workplace became really stressful," "felt like I had no choice"), but the retrieval model correctly mapped this to the legal concept of voluntary leaving and just cause under EI law.

**Recall Quality**
- *Are semantically similar but legally distinct decisions appropriately distinguished (e.g., CPP retirement vs. CPP disability)?*
  - The results are all EI cases concerning voluntary leaving, which indicates the system correctly distinguished this issue from other EI issues such as misconduct, availability for work, or hours of insurable employment. No unrelated benefit programs (CPP or OAS) appeared in the results, suggesting the semantic search model successfully separated similar language across different tribunal jurisdictions.

#### Stage 2: Cross-encoder Reranking

**Ranking Accuracy**
- *Does rank 1 represent the most fact-similar decision to the query?*
  - The top-ranked decision appears consistent with the scenario, as it involves a claimant who left their job and sought EI benefits, requiring the tribunal to determine whether the claimant had just cause for leaving.
- *Are decisions with similar facts but opposite outcomes both surfaced (so the user sees the full legal landscape)?*
  - Yes, the appeal was allowed in one of the cases.

**Score Calibration**
- *Do higher-scoring decisions feel meaningfully more relevant and/or different than lower-scoring ones?*
  - The similarity scores were very close, suggesting that the model viewed all five decisions as highly relevant.

**Length of Time**
- *How long does it take to retrieve and rank decisions?*
  - 7 seconds

#### Stage 3: Case Card Generation

**Accuracy**
- *Does the Issue section correctly identify the legal question decided in the source decision?*
  - The case card correctly identifies the legal issue as whether the appellant was entitled to EI benefits after voluntarily leaving his job and whether he had "just cause" under the Employment Insurance Act.
- *Do the Key Facts reflect facts actually present in the decision text, with no hallucinated details?*
  - The key facts accurately summarize the claimant's reasons for leaving, including workplace stress, long hours, and the need to relocate to care for his disabled wife. These facts reflect the tribunal's analysis of the claimant's personal circumstances.
- *Does the Test Applied section correctly describe the legal standard used?*
  - The test applied is correctly described. The summary explains that a claimant who voluntarily leaves employment must show they had no reasonable alternative to leaving, which is the legal standard used to determine "just cause" under EI law.
- *Does the Outcome section accurately reflect whether the appeal was allowed or dismissed?*
  - The outcome is also accurate. The tribunal dismissed the appeal because the claimant did not demonstrate that quitting was his only reasonable option, noting that alternatives such as requesting leave, seeking accommodations, or looking for other work were available.

**Plain Language Accessibility**
- *Can a self-represented litigant with no legal background understand the case card without needing to look up terminology?*
  - The case card is clear and accessible for a non-lawyer and the explanation of "just cause" is written in simple terms.
- *Is the summary concise enough to be skimmed quickly (a few sentences per section)?*
  - The structure (Issue, Key Facts, Test Applied, Outcome) makes the decision easy to scan.

**Faithfulness to Source**
- *Does the case card avoid adding legal conclusions not present in the original decision?*
  - The summary appears faithful to the tribunal decision. It does not introduce new legal conclusions beyond the reasoning described in the decision and accurately reflects the tribunal's explanation that personal reasons alone do not necessarily meet the EI just-cause standard.

**Length of Time**
- *How long does it take to generate the summary card?*
  - 11 seconds

#### Fast Mode Comparison Criteria

**Length of Time**
- *How long does it take to retrieve and rank decisions?* — 5 seconds
- *How long does it take to generate the summary card?* — 10 seconds

**Result Relevance**
- *Does the top case returned in fast mode address the same legal issue as the top case in regular mode?*
  - Yes. The top case in both modes addresses whether a claimant who voluntarily left their job had "just cause" under the Employment Insurance Act.
- *Does the fast mode still return factually similar decisions to the user's scenario?*
  - Yes. All three cases involve claimants who quit their jobs and argued they had valid reasons for leaving, requiring the tribunal to assess whether they had no reasonable alternative to quitting.
- *Are the three cases returned in fast mode included in the results of the regular mode?*
  - Yes. All three fast-mode cases were also included in the regular-mode results.
- *Are the three cases returned in fast mode all reasonably relevant, or does the reduced candidate set introduce weaker results?*
  - All three decisions remain highly relevant to the scenario. The reduced candidate set did not introduce any clearly weaker or unrelated cases.

**Coverage of Legal Landscape**
- *Does the reduced list of three cases omit any clearly important or highly relevant decision that appeared in regular mode?*
  - No clearly essential decisions appear to be omitted.
- *If the regular mode showed cases with different outcomes, does fast mode still surface that variation?*
  - No, the fast mode only showed cases where the appeal was dismissed.

**Summary Quality**
- *Is the summary generated in fast mode as accurate as the summary in regular mode?*
  - Yes. The fast-mode summary appears equally accurate and reflects the tribunal's reasoning.
- *Does the fast-mode summary still correctly describe: the legal issue, the key facts, the test applied, and the outcome?*
  - Yes. The summary correctly identifies the voluntary leaving issue, the reasons the claimant gave for quitting, the "no reasonable alternative" test, and the dismissal of the appeal.
- *Is the fast-mode summary clear and understandable for a non-lawyer, similar to the regular mode case card?*
  - Yes. The summary is written in plain, accessible language, and the structured format makes the decision easy for a non-lawyer to understand.

#### Comparison against a legal database search: CanLII

**Initial Search:** "quit job stress EI denied"

The results of this search were all decisions from the Social Security Tribunal of Canada relating to Employment Insurance claims involving voluntary leaving. For example, cases such as CB v. Canada Employment Insurance Commission focused on whether the claimant had a reasonable alternative to quitting, while B.S. v. Canada Employment Insurance Commission involved a claimant who left their job due to workplace stress and sought EI benefits. Other cases similarly addressed situations where individuals resigned for health or stress related reasons and the Tribunal assessed whether those reasons amounted to "just cause." The excerpts consistently referenced key concepts such as quitting, stress, reasonable alternatives, and disqualification from EI benefits. Overall, the results were highly relevant and closely aligned with the scenario.

**Refined Search:** "EI voluntary leaving just cause tribunal stress workplace"

The results were all Social Security Tribunal decisions specifically focused on the legal issue of voluntary leaving and "just cause" under the Employment Insurance Act. For example, cases such as S.T. v. Canada Employment Insurance Commission and C.W. v. Canada Employment Insurance Commission explicitly addressed whether the claimant had just cause for leaving their employment, including situations involving workplace harassment or intolerable working conditions. The excerpts consistently referenced the legal test of whether the claimant had no reasonable alternative to leaving and directly engaged with the concept of just cause. Overall, the results were highly targeted and clearly framed within the correct legal standard.

**Reflection:**
Comparing the two searches, the refined search produced more precise and legally focused results than the initial plain language search. While the initial search successfully identified relevant cases involving stress and quitting, it required the user to infer the underlying legal test. In contrast, the refined search directly incorporated key legal terminology such as "just cause" and "voluntary leaving," which narrowed the results to decisions explicitly applying the correct legal framework. This demonstrates that, on CanLII, using accurate legal language can significantly improve the precision of results. However, it also highlights a limitation for non-expert users, who may not know the correct legal terms to use, whereas the AI system is able to translate everyday language into the appropriate legal concepts automatically.

---

### Scenario 4: Generic Benefit Denial

> I applied for a government benefit but it was denied. I thought I met the requirements, but they said I didn't qualify.

#### Stage 1: Semantic Search (Bi-encoder Retrieval)

**Relevance of Candidate Set**
- *Do the top retrieved decisions relate to the user's described situation?*
  - Yes. The retrieved decisions all involve tribunal disputes about eligibility for government benefits, which matches the prompt describing a benefit denial. The results include cases from different Social Security Tribunal programs, including Employment Insurance and benefits administered by the Minister of Employment and Social Development (such as CPP or OAS). This reflects the broad nature of the prompt, which did not specify a particular benefit program.
- *Does the tool correctly handle colloquial/plain-language queries (Ex: "my boss fired me unfairly" → EI appeals)?*
  - Yes. Although the query uses very general language ("government benefit," "didn't qualify"), the model successfully retrieved cases about benefit eligibility disputes before the tribunal.

**Recall Quality**
- *Are semantically similar but legally distinct decisions appropriately distinguished (e.g., CPP retirement vs. CPP disability)?*
  - The results include cases from multiple benefit programs, which is appropriate given the prompt's general wording. This suggests the system recognized the query as relating to tribunal decisions about eligibility for benefits, rather than limiting results to a single program area. No unrelated types of decisions appeared, indicating the retrieval model still focused on relevant SST benefit disputes.

#### Stage 2: Cross-encoder Reranking

**Ranking Accuracy**
- *Does rank 1 represent the most fact-similar decision to the query?*
  - The top-ranked decisions appear consistent with the scenario, as they involve claimants who applied for benefits and were denied eligibility, requiring the tribunal to assess whether the decision was correct. Because the prompt is very general, the reranking stage appropriately surfaced cases across different benefit programs rather than focusing on a single legal test.
- *Are decisions with similar facts but opposite outcomes both surfaced (so the user sees the full legal landscape)?*
  - No, all of the appeals were dismissed.

**Score Calibration**
- *Do higher-scoring decisions feel meaningfully more relevant and/or different than lower-scoring ones?*
  - The similarity scores were again very close, indicating that the model viewed all retrieved cases as highly relevant.

**Length of Time**
- *How long does it take to retrieve and rank decisions?*
  - 27 seconds

#### Stage 3: Case Card Generation

**Accuracy**
- The case card correctly identifies the legal issue as whether the appeal regarding CPP death and survivor benefits should be dismissed because the deceased contributor did not meet the minimum CPP contribution requirements.
- The key facts accurately summarize the situation: the contributor had nine years of contributions, while the CPP required at least ten years or one-third of the contributory period (12 years). The applicant's argument for partial benefits is also accurately reflected.
- The test applied is described correctly. The summary explains that the tribunal must dismiss an appeal if it has no reasonable chance of success and must apply the statutory CPP contribution rules.
- The outcome is accurately reported: the appeal was dismissed because the contributor did not meet the contribution threshold and the tribunal had no authority to grant partial benefits outside the law.

**Plain Language Accessibility**
- The case card is clear and understandable for a non-lawyer. The explanation of the contribution requirement and why the appeal failed is written in simple terms, and the structured format makes the decision easy to skim.

**Faithfulness to Source**
- The summary appears faithful to the tribunal decision and does not introduce legal reasoning beyond what is described in the decision. It accurately reflects the tribunal's explanation that it cannot override statutory eligibility requirements for fairness reasons.

**Length of Time**
- *How long does it take to generate the summary card?*
  - 16 seconds

#### Fast Mode Comparison Criteria

**Length of Time**
- *How long does it take to retrieve and rank decisions?* — 6 seconds
- *How long does it take to generate the summary card?* — 18 seconds

**Result Relevance**
- *Does the top case returned in fast mode address the same legal issue as the top case in regular mode?*
  - Yes. The top case in both modes addresses the eligibility requirements for CPP death and survivor benefits, specifically whether the contributor met the minimum contribution threshold.
- *Does the fast mode still return factually similar decisions to the user's scenario?*
  - Yes. The fast-mode results still include cases where claimants were denied government benefits because they did not meet statutory eligibility requirements, which matches the user's prompt about being denied a benefit.
- *Are the three cases returned in fast mode included in the results of the regular mode?*
  - Yes. All three cases returned in fast mode were also included in the regular-mode results.
- *Are the three cases returned in fast mode all reasonably relevant, or does the reduced candidate set introduce weaker results?*
  - All three cases remain reasonably relevant to the prompt. Although they involve different benefit programs (CPP and EI), they all address eligibility disputes regarding government benefits.

**Coverage of Legal Landscape**
- *Does the reduced list of three cases omit any clearly important or highly relevant decision that appeared in regular mode?*
  - No clearly critical decisions appear to be omitted. However, reducing the list from five to three cases slightly reduces the range of examples across benefit programs.
- *If the regular mode showed cases with different outcomes, does fast mode still surface that variation?*
  - Neither version had variation in the outcomes.

**Summary Quality**
- *Is the summary generated in fast mode as accurate as the summary in regular mode?*
  - Yes. The fast-mode summary accurately reflects the tribunal's reasoning and outcome.
- *Does the fast-mode summary still correctly describe: the legal issue, the key facts, the test applied, and the outcome?*
  - Yes. The summary correctly identifies the CPP contribution requirement issue, the relevant facts about the contributor's years of contributions, the legal standard for summary dismissal, and the dismissal of the appeal.
- *Is the fast-mode summary clear and understandable for a non-lawyer, similar to the regular mode case card?*
  - Yes. The summary is written in clear and accessible language, and the structured format makes the decision easy for a non-lawyer to understand.

#### Comparison against a legal database search: CanLII

**Initial Search:** "government benefit denied didn't qualify tribunal"

The results were primarily Social Security Tribunal decisions relating to Employment Insurance eligibility, rather than a specific benefit program. For example, cases such as MC v. Canada Employment Insurance Commission and Canada Employment Insurance Commission v. RL focused on whether claimants met the requirements to qualify for EI benefits, including issues related to qualifying periods and delays in applying. While these cases involved disputes over benefit eligibility, they were not consistently tied to a single legal framework or type of benefit, reflecting the broad nature of the search. Overall, the results were somewhat relevant but lacked precision, as they captured a wide range of benefit related disputes rather than a clearly defined legal issue.

**Refined Search:** "CPP OAS EI eligibility tribunal decision requirements not met"

The results included a mix of Social Security Tribunal decisions across different benefit programs, including Employment Insurance and Old Age Security, as well as some Federal Court decisions. For example, cases such as MG v. Canada Employment Insurance Commission focused on EI eligibility requirements, while AM v. Minister of Employment and Social Development addressed OAS residency and eligibility for partial pensions. Other results referenced the Tribunal's authority to reconsider decisions and apply statutory eligibility criteria. Overall, while the results were related to benefit eligibility disputes, they remained broad and covered multiple legal frameworks rather than a single clearly defined issue.

**Reflection:**
Comparing the initial and refined searches, both produced relevant but broad results due to the general nature of the scenario. The refined search introduced more formal legal terminology and explicitly referenced multiple benefit programs, but this further widened the scope rather than narrowing it. As a result, neither search produced highly targeted results tied to a specific legal test or statutory framework. This demonstrates that when a query is overly general, even the use of legal language may not improve precision on CanLII. In contrast, the AI system is better able to interpret vague inputs and still generate focused, relevant outputs by identifying the most applicable legal context without requiring the user to specify it.

---

### Testing Part 2: Testing the Limits

*Disclaimer: The validation testing questions and the first 2 of the prompts were generated with ChatGPT based on the GPT-5.3 model (OpenAI). Answers for the testing criteria and reflection were refined using ChatGPT based on the GPT-5.3 model (OpenAI) and edited via Grammarly.*

In order to incorporate the feedback we were given to intentionally get the system to return bad results, we decided to incorporate spelling and grammatical errors as well as vague or confusing phrasing in the prompts. Two of the prompts were generated with the GPT 5.4 Model with the system message set to "Pretend you are a middle-aged adult with average IQ and no legal background or knowledge." We then manually added spelling and grammatical errors to the provided prompts before we inputted it into the system. For the other two prompts, we had real people provide them for us so that we could also consider whether AI generated examples are a good way to test this system.

---

### Scenario 1: CPP Disability — Late MQP

> they told me my conditon got bad to late so i dont qualify because of some deadline, even though im not able to work now.

**Issue Identification**
- *Did the system identify the correct legal issue, or did it pick the wrong program (CPP vs EI vs OAS)?*
  - The system correctly identified the relevant legal issue despite the absence of legal terminology.
  - It correctly identified CPP disability and specifically, the Minimum Qualifying Period (MQP).
  - It correctly inferred that the program was CPP and the issue was disability timing/MQP.

**Relevance of Output**
- *Are the returned cases still meaningfully related to the scenario?*
  - The returned cases were highly relevant and directly addressed the timing requirement for CPP disability eligibility.
  - All of the returned cases were CPP disability or MQP/timing-related.
  - The summary case directly matched the "too late" disability issue.

**Handling of Ambiguity**
- *Does the system handle unclear prompts reasonably — give an overconfident but weak answer?*
  - The system handled ambiguity effectively and was able to infer the correct legal framework from minimal and imprecise input.
  - The prompt was vague, emotionally framed, and had no legal terms.
  - The system did NOT go generic, did NOT mix programs, and confidently picked the correct doctrine.

**Summary Behavior**
- *Does the summary reflect the user's situation or ignore key parts of the prompt?*
  - The summary accurately reflects the legal issue and reasoning, though it places greater emphasis on the MQP requirement than on the claimant's current inability to work.
  - It clearly explained the MQP deadline and why the claim failed.
  - It also matched the prompt's core issue by explicitly addressing being "too late."
  - However, the prompt says "I'm not able to work now" but the summary only focused on timing, not current inability.
- *Does it oversimplify complex scenarios?*
  - The explanation did not oversimplify the issue; it just focused on the core legal rule, even if that meant leaving out some of the user's perspective.

**Failure Type:** None — Actually handled well

---

### Scenario 2: EI — Conflicting Letters

> I got 2 letters 1 says "not eligble" and the next says my amount will be adjuted. They dont agree with each other so i dont know which one is the real one.

**Issue Identification**
- *Did the system identify the correct legal issue, or did it pick the wrong program (CPP vs EI vs OAS)?*
  - The system identified a related but misaligned legal issue that does not fully capture the user's core concern.
  - The system identified the issue as a procedural error relating to whether the tribunal decided matters outside the scope of the original Employment Insurance decision.
  - Since the prompt was about receiving conflicting letters and not knowing which one represents the actual decision, the system should have identified the issue as one of decision clarity, specifically determining which communication is legally operative for appeal purposes.

**Relevance of Output**
- *Are the returned cases still meaningfully related to the scenario?*
  - The output is only partially relevant and did not closely match the factual scenario presented.
  - The system returned cases involving procedural errors in Employment Insurance appeals, specifically where tribunals considered issues beyond those properly before them.
  - These cases involved some decision-related confusion but did not directly address conflicting letters or uncertainty about which decision prevails.
  - The system should have retrieved cases dealing with multiple or unclear Commission communications and reconsideration decisions.

**Handling of Ambiguity**
- *Does the system handle unclear prompts reasonably — give an overconfident but weak answer?*
  - The system handled ambiguity in a legally structured but not practically responsive manner.
  - The system handled the vague and informal prompt by selecting a related Employment Insurance framework and avoiding confusion with other programs.
  - However, it defaulted to a more technical and abstract legal issue rather than addressing the user's practical confusion.
  - The system should have interpreted the ambiguity in a more user-centered way by focusing on the real-world issue of conflicting information.

**Summary Behavior**
- *Does the summary reflect the user's situation or ignore key parts of the prompt?*
  - The summary is legally coherent but did not directly answer the user's question and failed to fully reflect the user's situation.
  - The summary partially reflected the user's situation by recognizing that multiple letters created confusion.
  - However, it did not directly address the user's core concern about which letter represents the actual or operative decision. Instead, it reframed the issue into a procedural error about tribunal jurisdiction, which shifted away from the user's practical question.
  - The system should have focused on explaining how conflicting communications are interpreted and which decision is legally controlling.
- *Does it oversimplify complex scenarios?*
  - The summary did not significantly oversimplify the legal issue but instead redirected it into a more abstract and less user-focused explanation.

**Failure Type:** Partial relevance — The system identified a related procedural issue but did not directly address the user's core concern about conflicting decisions and it failed to engage with the key factual problem of determining which communication was legally operative.

---

### Scenario 3: EI — Missed Shift and Scheduling

> I couldnt go to my work because my daughters daycare closed. My boss said if I dont come then I am not reliable. After I didnt go they did not schedele me again and it has been 3 weeks.

**Issue Identification**
- *Did the system identify the correct legal issue, or did it pick the wrong program (CPP vs EI vs OAS)?*
  - The system identified the wrong legal issue by treating the situation as quitting rather than a potential loss of employment or dismissal.
  - The system identified the issue as voluntary leaving with "just cause" under EI.
  - However, the user did not say they quit; they described missing one shift and then not being scheduled again.
  - The correct issue should have been whether this situation amounts to dismissal, loss of employment, or possibly misconduct, not voluntary leaving.

**Relevance of Output**
- *Are the returned cases still meaningfully related to the scenario?*
  - The output is only partially relevant and does not accurately reflect the user's employment situation.
  - The cases returned focused on voluntary leaving due to childcare conflicts, which is related at a surface level.
  - However, the user's situation involves being informally cut off from work rather than choosing to leave, which is legally distinct.
  - The system should have retrieved cases involving dismissal, reduced hours, or employment relationship breakdown, not primarily quitting cases.

**Handling of Ambiguity**
- *Does the system handle unclear prompts reasonably — give an overconfident but weak answer?*
  - The system handled ambiguity by forcing a single interpretation, rather than considering alternative legal characterizations.
  - The system recognized the childcare element and placed the issue within the EI framework.
  - However, it resolved the ambiguity by assuming the claimant chose to leave the job, rather than considering multiple possible interpretations of the situation.
  - The system should have explored whether the situation involved dismissal, constructive dismissal, or availability issues.

**Summary Behavior**
- *Does the summary reflect the user's situation or ignore key parts of the prompt?*
  - The summary reflected part of the user's situation by incorporating the childcare issue and conflict with work schedule.
  - However, it assumed that the claimant voluntarily left the job, which is not stated in the prompt and is a key mischaracterization.
  - The summary should have addressed whether the claimant was effectively dismissed or stopped being scheduled, rather than framing it as quitting.
- *Does it oversimplify complex scenarios?*
  - The summary oversimplified the scenario by reducing a potentially complex employment situation into a straightforward voluntary leaving case.

**Failure Type:** Missed key facts — The system mischaracterized the situation as voluntary leaving and ignored key facts about the employment relationship.

---

### Scenario 4: EI — Benefit Reassessment

> Randomly I got a review notice. Nothing much changed recently but they said my situation needs to be re-checked and then my beneft amount changed but i dont get what changed to make that happn.

**Issue Identification**
- *Did the system identify the correct legal issue, or did it pick the wrong program (CPP vs EI vs OAS)?*
  - The system identified a related but overly technical procedural issue rather than the user's concern about benefit reassessment.
  - The system identified the issue as a procedural/legal error involving changing benefit calculations during an appeal.
  - However, the prompt was about a review or reassessment of benefits and not understanding why their amount changed.
  - The correct issue should have been benefit reassessment, review decisions, or recalculation of entitlement, not appellate procedural error.

**Relevance of Output**
- *Are the returned cases still meaningfully related to the scenario?*
  - The output is partially relevant but does not closely match the user's situation.
  - The cases involved changes to benefit calculations and Commission decisions, which is somewhat related.
  - However, they focused on legal errors during appeals, not general reassessment or review notices.
  - The system should have retrieved cases about benefit recalculation, overpayments, or reassessments following reviews.

**Handling of Ambiguity**
- *Does the system handle unclear prompts reasonably — give an overconfident but weak answer?*
  - The system handled ambiguity in a legally structured but overly technical and misdirected way.
  - The system recognized that the issue involved benefits being changed and stayed within the EI framework.
  - However, it interpreted the ambiguity in a highly technical way, focusing on appeal-stage legal errors rather than everyday reassessment.
  - The system should have taken a more user-centered approach by focusing on why benefits change after reviews.

**Summary Behavior**
- *Does the summary reflect the user's situation or ignore key parts of the prompt?*
  - The summary was detailed but did not directly address the user's situation and focused on an overly technical issue.
  - The summary reflected part of the situation by addressing changes to benefit amounts and confusion arising from those changes.
  - However, it focused on a specific legal issue involving changes during an appeal, rather than explaining why a claimant might receive a review notice or reassessment.
  - The summary should have explained how benefit amounts can change following a review and what factors might trigger that reassessment.
- *Does it oversimplify complex scenarios?*
  - The explanation did not oversimplify the law; it just shifted into a more complex and less relevant legal scenario.

**Failure Type:** Partial relevance — The system focused on a technical appeal issue instead of the user's concern about benefit reassessment and changes, and failed to address the key factual question of why the benefit amount changed despite no apparent change in circumstances.

---

### Reflection

The AI-generated prompts were more structured and internally consistent, and in some cases this allowed the system to correctly identify the legal issue and return highly relevant results. However, this was not uniform — one AI-generated prompt still resulted in a partially misaligned response. In contrast, the prompts based on real user inputs were more ambiguous and reflective of how individuals naturally describe their situations, which appeared to increase the likelihood of misinterpretation. Across the scenarios with failures, the failure types were notably consistent, with the system frequently producing partially relevant outputs and missing key aspects of the user's concern. Importantly, spelling and grammatical errors in the prompts did not appear to significantly affect performance, suggesting that the system is robust to surface-level language issues. Overall, these results indicate that the system's primary limitation lies not in identifying a general legal framework or understanding imperfect language, but in accurately interpreting and responding to the underlying intent of more complex and ambiguous user inputs.